# Session 9: Working with Dates & Time

**Course:** Python for Data Engineering  
**Phase 2:** Data Handling & Transformation

**What we'll cover:**
- Python's `datetime` module
- Pandas datetime operations
- Time-based transformations and feature engineering

**Data files:** `data/orders_with_dates.csv` — same customers and products, now with timestamps spanning a full year.

**Note:** This session is demo-heavy. Follow along by running each cell.

In [ ]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta

---

## 1. Python's datetime Module

Before pandas, let's understand the built-in datetime. You'll use this in file naming, log timestamps, scheduling, etc.

In [ ]:
# Current date and time
now = datetime.now()
print(f"Now: {now}")
print(f"Date: {now.date()}")
print(f"Time: {now.time()}")
print(f"Year: {now.year}, Month: {now.month}, Day: {now.day}")
print(f"Hour: {now.hour}, Minute: {now.minute}")

In [ ]:
# Creating a specific datetime
pipeline_start = datetime(2024, 1, 15, 9, 30, 0)
print(f"Pipeline started: {pipeline_start}")

# Parsing a string into datetime — strptime
date_str = "2024-01-15 14:30:00"
parsed = datetime.strptime(date_str, "%Y-%m-%d %H:%M:%S")
print(f"Parsed: {parsed}")

# Formatting datetime to string — strftime
print(f"Formatted: {parsed.strftime('%d/%m/%Y')}")
print(f"File name: output_{parsed.strftime('%Y%m%d')}.csv")

In [ ]:
# Common format codes
# %Y = 2024, %m = 01, %d = 15
# %H = 14, %M = 30, %S = 00
# %A = Monday, %B = January

print(parsed.strftime("%A, %B %d, %Y"))     # Monday, January 15, 2024
print(parsed.strftime("%Y-%m-%d"))            # 2024-01-15
print(parsed.strftime("%d-%b-%Y %I:%M %p"))   # 15-Jan-2024 02:30 PM

In [ ]:
# timedelta — date arithmetic

start = datetime(2024, 1, 15, 9, 0, 0)
end = datetime(2024, 1, 15, 11, 30, 0)

duration = end - start
print(f"Duration: {duration}")
print(f"In seconds: {duration.total_seconds()}")
print(f"In minutes: {duration.total_seconds() / 60}")

# Add/subtract time
tomorrow = now + timedelta(days=1)
last_week = now - timedelta(weeks=1)
two_hours_later = now + timedelta(hours=2)

print(f"\nNow: {now.strftime('%Y-%m-%d')}")
print(f"Tomorrow: {tomorrow.strftime('%Y-%m-%d')}")
print(f"Last week: {last_week.strftime('%Y-%m-%d')}")

---

## 2. Pandas Datetime — pd.to_datetime()

In pandas, you convert date columns with `pd.to_datetime()`. Once converted, you get access to the `.dt` accessor for extracting parts.

In [ ]:
# Load orders with timestamps
df = pd.read_csv("data/orders_with_dates.csv")
print(df.dtypes)
print()
df.head()

In [ ]:
# order_timestamp is a string — convert to datetime
df["order_timestamp"] = pd.to_datetime(df["order_timestamp"])
print(df.dtypes)
print()
df.head()

In [ ]:
# Extract date parts using .dt accessor
df["year"] = df["order_timestamp"].dt.year
df["month"] = df["order_timestamp"].dt.month
df["day"] = df["order_timestamp"].dt.day
df["day_name"] = df["order_timestamp"].dt.day_name()
df["hour"] = df["order_timestamp"].dt.hour
df["quarter"] = df["order_timestamp"].dt.quarter
df["week"] = df["order_timestamp"].dt.isocalendar().week

df[["order_id", "order_timestamp", "year", "month", "quarter", "day_name", "hour"]]

In [ ]:
# Filtering by date range

# Orders from Q1 2024
q1 = df[df["order_timestamp"].between("2024-01-01", "2024-03-31")]
print(f"Q1 orders: {len(q1)}")
print(q1[["order_id", "customer", "product", "order_timestamp"]])

# Orders in the last 6 months of 2024
h2 = df[df["order_timestamp"] >= "2024-07-01"]
print(f"\nH2 orders: {len(h2)}")

---

## 3. Time-Based Aggregation

Grouping by month, quarter, or week — common in pipeline reporting and dashboards.

In [ ]:
# Add total column
df["total"] = df["quantity"] * df["unit_price"]

# Monthly revenue
monthly = df.groupby(df["order_timestamp"].dt.to_period("M")).agg(
    orders=("order_id", "count"),
    revenue=("total", "sum"),
).reset_index()

monthly.columns = ["month", "orders", "revenue"]
print("Monthly summary:")
print(monthly)

In [ ]:
# Quarterly revenue
quarterly = df.groupby("quarter").agg(
    orders=("order_id", "count"),
    revenue=("total", "sum"),
    unique_customers=("customer", "nunique"),
).reset_index()

print("Quarterly summary:")
print(quarterly)

In [ ]:
# resample — pandas built-in time grouping (needs datetime index)
df_ts = df.set_index("order_timestamp")

# Monthly revenue using resample
monthly_rs = df_ts["total"].resample("ME").sum()
print("Monthly revenue (resample):")
print(monthly_rs)

---

## 4. Time-Based Feature Engineering

Creating new columns derived from dates — useful for downstream analysis and ML pipelines.

In [ ]:
# Is it a weekend order?
df["is_weekend"] = df["order_timestamp"].dt.dayofweek >= 5

# Morning (before noon) or afternoon?
df["time_of_day"] = df["hour"].apply(lambda h: "morning" if h < 12 else "afternoon")

# Days since first order (recency)
df["days_since_first"] = (df["order_timestamp"] - df["order_timestamp"].min()).dt.days

df[["order_id", "order_timestamp", "day_name", "is_weekend", "time_of_day", "days_since_first"]]

In [ ]:
# Days between orders per customer — inter-order gap

df_sorted = df.sort_values(["customer", "order_timestamp"])
df_sorted["prev_order"] = df_sorted.groupby("customer")["order_timestamp"].shift(1)
df_sorted["days_between_orders"] = (df_sorted["order_timestamp"] - df_sorted["prev_order"]).dt.days

print("Order gaps per customer:")
print(df_sorted[["customer", "order_timestamp", "prev_order", "days_between_orders"]].dropna())

---

## Lab: Monthly Sales Report with Time Features

Using `data/orders_with_dates.csv`, build a report that answers:

1. Load the data, convert timestamps, add a `total` column
2. Which **quarter** had the highest revenue?
3. Which **day of the week** gets the most orders?
4. For each **customer**, calculate: first order date, last order date, total orders, avg days between orders
5. Create a `month_name` column (e.g., "January") and show revenue per month name
6. Save the enriched DataFrame (with all new columns) to `data/orders_enriched.csv`

In [ ]:
import pandas as pd

# Your code here


---

## Summary

| Topic | Key Takeaway |
|-------|--------------|
| `datetime` module | `strptime` to parse, `strftime` to format, `timedelta` for arithmetic |
| `pd.to_datetime()` | Convert string columns to datetime |
| `.dt` accessor | Extract year, month, day, hour, day_name, quarter, etc. |
| Date filtering | `df[df['col'].between('start', 'end')]` |
| Time aggregation | GroupBy with `.dt.to_period()` or `.resample()` |
| Feature engineering | is_weekend, time_of_day, days_between_orders |

**Key patterns:**
- Always convert date strings to datetime first — don't do string comparison on dates
- `.dt` accessor is your friend for extracting parts
- resample needs a datetime index, groupby doesn't
- Time-based features (recency, gaps, day of week) are very common in DE pipelines

**Next session:** ETL Pipeline Design — putting everything together into structured Extract → Transform → Load pipelines.